# V2.5.1.1 — Re-test high_volatility_prob under the TUNED XGBoost

The original V2.5.1 experiment found that adding `high_volatility_prob` made the
model slightly WORSE — but it was tested on an **untuned** XGBoost (V2.5.2 params,
10-trial Optuna). Now that XGBoost is properly tuned (V2.5.3, 30-trial Optuna),
we re-run the same controlled experiment to see if the feature helps a model
that is actually well-configured.

- Data: `V2.5.1_15min_Risk_Enhanced_Dataset.csv` (105,193 rows)
- Split: same chronological 80/20
- Hyperparameters: **V2.5.3 best params** (MAE loss, 2000 trees)
- Only difference: baseline (49 features) vs enhanced (49 + high_volatility_prob = 50)
- `price_roll_std_6h` and `is_high_volatility` excluded from BOTH (leakage).

In [1]:
import numpy as np
import pandas as pd
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# Fix: Chinese Windows GBK -> sklearn HTML repr UnicodeDecodeError; force text display.
import sklearn
sklearn.set_config(display='text')

In [2]:
df = pd.read_csv('../data/convertData/V2.5.1_15min_Risk_Enhanced_Dataset.csv')
df['datetime'] = pd.to_datetime(df['datetime'], utc=True).dt.tz_convert('Europe/Helsinki')
df = df.sort_values('datetime').reset_index(drop=True)
print('Shape:', df.shape)
print('has high_volatility_prob:', 'high_volatility_prob' in df.columns)

Shape: (105193, 54)
has high_volatility_prob: True


In [3]:
# build the two feature matrices (controlled experiment)
risk_cols = ['price_roll_std_6h', 'is_high_volatility', 'high_volatility_prob']

baseline_cols = [c for c in df.columns if c not in ['price', 'datetime'] + risk_cols]
enhanced_cols = baseline_cols + ['high_volatility_prob']

X_base = df[baseline_cols]   # 49 features
X_enh  = df[enhanced_cols]   # 50 features
y = df['price']
print(f'baseline: {len(baseline_cols)} features | enhanced: {len(enhanced_cols)} features')

# same chronological 80/20 split for both feature sets
n = len(df)
test_size = int(n * 0.20)
train_end = n - test_size

X_base_train, X_base_test = X_base.iloc[:train_end], X_base.iloc[train_end:]
X_enh_train,  X_enh_test  = X_enh.iloc[:train_end],  X_enh.iloc[train_end:]
y_train, y_test = y.iloc[:train_end], y.iloc[train_end:]
print(f'Train: {X_base_train.shape[0]}  Test: {X_base_test.shape[0]}')

baseline: 49 features | enhanced: 50 features
Train: 84155  Test: 21038


In [4]:
# V2.5.3 best hyperparameters (30-trial Optuna, MAE loss, 2000 trees)
tuned = dict(
    objective='reg:absoluteerror', n_estimators=2000,
    learning_rate=0.00982714905428372, max_depth=12, min_child_weight=31,
    subsample=0.7997314659075123, colsample_bytree=0.9982960915995492,
    reg_lambda=0.012943440208283537, reg_alpha=0.43805234879252597,
    random_state=42,
)

def train_eval(Xtr, ytr, Xte, yte):
    """Train one tuned XGBoost and return (MAE, RMSE, R2)."""
    m = XGBRegressor(**tuned, verbosity=0)
    m.fit(Xtr, ytr)
    p = m.predict(Xte)
    return (mean_absolute_error(yte, p),
            np.sqrt(mean_squared_error(yte, p)),
            r2_score(yte, p))

base = train_eval(X_base_train, y_train, X_base_test, y_test)
enh  = train_eval(X_enh_train,  y_train, X_enh_test,  y_test)
print('Both models trained (tuned XGBoost).')

Both models trained (tuned XGBoost).


In [5]:
comp = pd.DataFrame(
    {'XGBoost baseline (49)': base, 'XGBoost +risk (50)': enh},
    index=['MAE', 'RMSE', 'R2']).T.round(4)
print(comp)

delta = enh[0] - base[0]
print('\nMAE delta (enhanced - baseline): %+.4f' % delta)
if delta < 0:
    print('Verdict: high_volatility_prob HELPS under tuned XGBoost')
elif delta > 0:
    print('Verdict: high_volatility_prob still HURTS even under tuned XGBoost')
else:
    print('Verdict: no difference')

print('\nReference (original V2.5.1, untuned XGBoost):')
print('  baseline MAE 2.7555 -> +risk MAE 2.7957 (delta +0.0402, worse)')

                          MAE    RMSE      R2
XGBoost baseline (49)  2.7079  8.1223  0.9725
XGBoost +risk (50)     2.7124  8.1593  0.9722

MAE delta (enhanced - baseline): +0.0045
Verdict: high_volatility_prob still HURTS even under tuned XGBoost

Reference (original V2.5.1, untuned XGBoost):
  baseline MAE 2.7555 -> +risk MAE 2.7957 (delta +0.0402, worse)


## Verdict — high_volatility_prob STILL does not help (tuned XGBoost)

| Model | MAE | RMSE | R² |
| ----- | --- | ---- | --- |
| XGBoost baseline (49) | **2.7079** | 8.1223 | 0.9725 |
| XGBoost +risk (50) | 2.7124 | 8.1593 | 0.9722 |

MAE delta: **+0.0045** (still slightly WORSE).

**Conclusion:** Even under the properly-tuned XGBoost (V2.5.3 params), the
`high_volatility_prob` feature does NOT help — it still hurts, though the harm is
much smaller than in the original untuned V2.5.1 experiment (+0.0045 vs +0.0402).

This is a **robust negative result**: the feature comes from a weak classifier
(recall 0.24 for high volatility), so it is "weak signal + noise". The problem was
NOT the model being undertuned — the feature itself does not carry enough signal.
**Decision: do NOT add high_volatility_prob.** Improvement direction remains
strengthening the classifier first (e.g. scale_pos_weight, more features).